# 第5章 総合演習：渋谷・原宿エリアのカフェ出店候補地を選ぶ / Capstone: Choosing a Café Location in Shibuya–Harajuku
参照用ノートブックです。`LANG` を選んで、すべてのセルを上から実行します。 / Choose `LANG` and run all cells.

配布データ：shibuya_harajuku_mesh.csv（第4章のデータと国勢調査の年齢別人口から作成）、tokyo23_wards.geojson、stations_tokyo23_2023.csv

In [ ]:
LANG = "ja"   # "ja" / "en"
BASE_URL = "https://raw.githubusercontent.com/YOUR_ACCOUNT/YOUR_REPO/main/data/"

In [ ]:
import os, subprocess, urllib.request, warnings; warnings.filterwarnings("ignore")
subprocess.run("pip install -q geopandas folium", shell=True)
for f in ["shibuya_harajuku_mesh.csv", "tokyo23_wards.geojson", "stations_tokyo23_2023.csv"]:
    if not os.path.exists(f):
        try: urllib.request.urlretrieve(BASE_URL + f, f)
        except Exception as e: print(f, "をアップロードしてください / please upload", e)
JP = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if not os.path.exists(JP):
    subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True)

## メッシュコードから区画を作る関数 / Grid-cell polygons from mesh codes

In [ ]:
import numpy as np
def mesh_sw(code):
    c=str(code); lat=int(c[0:2])/1.5; lon=int(c[2:4])+100
    lat+=int(c[4])*5/60; lon+=int(c[5])*7.5/60
    lat+=int(c[6])*0.5/60; lon+=int(c[7])*0.75/60
    dlat,dlon=0.5/60,0.75/60
    for d in c[8:]:
        dlat/=2; dlon/=2; k=int(d)
        if k in (3,4): lat+=dlat
        if k in (2,4): lon+=dlon
    return lat,lon,dlat,dlon
def mesh_center(code):
    lat,lon,dlat,dlon=mesh_sw(code); return lat+dlat/2, lon+dlon/2
def mesh_polygon(code):
    from shapely.geometry import box
    lat,lon,dlat,dlon=mesh_sw(code); return box(lon,lat,lon+dlon,lat+dlat)

## 5-2〜5-4 正規化・加重スコア・感度分析 / Normalization, weighted scores, sensitivity

In [ ]:
import pandas as pd, numpy as np
m=pd.read_csv("shibuya_harajuku_mesh.csv",dtype={"mesh_code":str})
pd.set_option("display.width", 200)
IND={"day_pop":1,"young_women":1,"inflow_school":1,"pop_foreign":1,"dist_station_m":-1,"station_passengers":1,"restaurants_500m":-1}
def norm(df,how="minmax"):
    n=pd.DataFrame(index=df.index)
    for c,s in IND.items():
        x=df[c].astype(float)
        v=(x-x.min())/(x.max()-x.min()) if how=="minmax" else x.rank(pct=True)
        n[c]=v if s>0 else 1-v if how=="minmax" else (1-v+1/len(x))
    return n
SC={"基本":{"day_pop":30,"young_women":20,"inflow_school":15,"pop_foreign":10,"dist_station_m":10,"station_passengers":5,"restaurants_500m":10},
    "人の多さ重視":{"day_pop":50,"young_women":10,"inflow_school":10,"pop_foreign":5,"dist_station_m":5,"station_passengers":20,"restaurants_500m":0},
    "競合回避":{"day_pop":20,"young_women":20,"inflow_school":10,"pop_foreign":5,"dist_station_m":10,"station_passengers":5,"restaurants_500m":30},
    "若い女性重視":{"day_pop":20,"young_women":40,"inflow_school":15,"pop_foreign":5,"dist_station_m":10,"station_passengers":5,"restaurants_500m":5}}
n=norm(m)
for k,w in SC.items():
    m["score_"+k]=sum(n[c]*w[c] for c in IND)/100
show=["mesh_code","nearest_station","dist_station_m","day_pop","young_women","inflow_school","pop_foreign","restaurants_500m","station_passengers"]
b=m.sort_values("score_基本",ascending=False)
print(b[show+["score_基本"]].head(10).to_string())
top5={k:set(m.nlargest(5,"score_"+k).mesh_code) for k in SC}
from collections import Counter
cnt=Counter(x for s in top5.values() for x in s); print(cnt.most_common(10))
for k in SC: print(k, m.nlargest(5,"score_"+k)[["nearest_station","dist_station_m"]].values.tolist())
nr=norm(m,"rank"); m["score_rank"]=sum(nr[c]*SC["基本"][c] for c in IND)/100
print("rank-based top5",m.nlargest(5,"score_rank")[["mesh_code","nearest_station"]].values.tolist())
print(m[list(IND)].corr().round(2))
print("school max share",(m.inflow_school.max()), m.inflow_school.quantile([.5,.9]).tolist())
m.to_csv("scored.csv",index=False)

## 図 / Figures

In [ ]:
import os, pandas as pd, numpy as np, geopandas as gpd, matplotlib
import matplotlib.pyplot as plt, matplotlib.patheffects as pe
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
J=LANG=="ja"; out=f"figures/{LANG}"; os.makedirs(out,exist_ok=True)
JP="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"; fm.fontManager.addfont(JP)
plt.rcParams.update({"font.family":fm.FontProperties(fname=JP).get_name(),"font.size":8,"savefig.dpi":300})
STEN={"渋谷":"Shibuya","表参道":"Omotesando","原宿":"Harajuku","明治神宮前":"Meiji-jingumae","代々木公園":"Yoyogi-koen","代々木八幡":"Yoyogi-Hachiman","神泉":"Shinsen","外苑前":"Gaiemmae","広尾":"Hiroo","参宮橋":"Sangubashi","北参道":"Kita-sando","代官山":"Daikanyama","国立競技場":"Kokuritsu-kyogijo","池尻大橋":"Ikejiri-ohashi","恵比寿":"Ebisu","駒場東大前":"Komaba-todaimae","代々木":"Yoyogi","千駄ケ谷":"Sendagaya","青山一丁目":"Aoyama-itchome"}
m=pd.read_csv("scored.csv",dtype={"mesh_code":str})
g=gpd.GeoDataFrame(m,geometry=[mesh_polygon(c) for c in m.mesh_code],crs=4326).to_crs(6677)
W=gpd.read_file("tokyo23_wards.geojson").to_crs(6677)
bb=g.total_bounds; W=W.clip(bb)
st=pd.read_csv("stations_tokyo23_2023.csv")
gs=gpd.GeoDataFrame(st,geometry=gpd.points_from_xy(st.lon,st.lat),crs=4326).to_crs(6677)
gs=gs[(gs.geometry.x.between(bb[0],bb[2]))&(gs.geometry.y.between(bb[1],bb[3]))]
halo=[pe.withStroke(linewidth=1.8,foreground="white")]
C5=["#f0f0f0","#c8c8c8","#969696","#636363","#252525"]
def base(ax,col,bins,labs,title,top=None,legend=True,lab_st=True,fs=6):
    cl=pd.cut(g[col],bins,labels=False,right=False,include_lowest=True)
    g.plot(ax=ax,color=[C5[int(c)] for c in cl],edgecolor="white",lw=0.3)
    W.boundary.plot(ax=ax,color="black",lw=0.6,ls="--")
    ax.scatter(gs.geometry.x,gs.geometry.y,s=16,facecolor="white",edgecolor="black",lw=0.9,zorder=5)
    if lab_st:
        for _,r in gs.iterrows(): ax.text(r.geometry.x+60,r.geometry.y+(-170 if r.station_name=="代々木八幡" else 50),r.station_name if J else STEN.get(r.station_name,r.station_name),fontsize=fs,path_effects=halo,zorder=6)
    if top is not None:
        for i,(_,r) in enumerate(top.iterrows(),1):
            c=r.geometry.centroid; ax.text(c.x,c.y,str(i),ha="center",va="center",fontsize=7,fontweight="bold",color="white",zorder=7,
                                           bbox=dict(boxstyle="circle,pad=0.15",fc="black",ec="white",lw=0.8))
    ax.set_xlim(bb[0],bb[2]); ax.set_ylim(bb[1],bb[3]); ax.set_axis_off()
    if legend: ax.legend(handles=[Patch(fc=c,ec="black",lw=0.4,label=l) for c,l in zip(C5,labs)],title=title,loc="lower left",bbox_to_anchor=(1.0,0.0),fontsize=6.3,title_fontsize=7,frameon=False)
# 5-1-1 daytime pop
fig,ax=plt.subplots(figsize=(4.5,3.4))
base(ax,"day_pop",[0,1000,2500,5000,10000,1e9],["1,000未満","1,000〜2,499","2,500〜4,999","5,000〜9,999","10,000以上"] if J else ["< 1,000","1,000–2,499","2,500–4,999","5,000–9,999","≥ 10,000"],"昼間人口（人）" if J else "Daytime population")
fig.savefig(f"{out}/fig5-1-1_area_daytime.png",bbox_inches="tight"); plt.show()
# 5-3-1 score map basic
top=g.nlargest(5,"score_基本")
fig,ax=plt.subplots(figsize=(4.5,3.4))
base(ax,"score_基本",[0,0.2,0.3,0.4,0.5,1.01],["0.2未満","0.2〜0.3","0.3〜0.4","0.4〜0.5","0.5以上"] if J else ["< 0.2","0.2–0.3","0.3–0.4","0.4–0.5","≥ 0.5"],"総合スコア" if J else "Composite score",top=top)
fig.savefig(f"{out}/fig5-3-1_score_map.png",bbox_inches="tight"); plt.show()
# 5-4-1 sensitivity 2x2
names={"基本":"Base","人の多さ重視":"Crowd-focused","競合回避":"Avoid competition","若い女性重視":"Young-women-focused"}
fig,axs=plt.subplots(2,2,figsize=(4.5,3.9))
for ax,k in zip(axs.flat,["基本","人の多さ重視","競合回避","若い女性重視"]):
    sc=g["score_"+k]; q=np.quantile(sc,[0.2,0.4,0.6,0.8]); g["_q"]=sc
    base(ax,"_q",[-1,*q,9],None,None,top=g.nlargest(5,"score_"+k),legend=False,lab_st=False)
    ax.set_title(k if J else names[k],fontsize=7.5)
fig.tight_layout(); fig.savefig(f"{out}/fig5-4-1_sensitivity.png",bbox_inches="tight"); plt.show()
print("ok")

## 5-5 インタラクティブ地図で候補地を示す / Interactive map of the candidates

In [ ]:
import folium
g4 = g.to_crs(4326)
m5 = folium.Map(location=[35.664, 139.702], zoom_start=15,
                tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
                attr='<a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank">地理院タイル</a>')
folium.Choropleth(geo_data=g4.__geo_interface__, data=g4, columns=["mesh_code", "score_基本"], key_on="feature.properties.mesh_code",
                  fill_color="Greys", fill_opacity=0.6, line_opacity=0.2,
                  legend_name="総合スコア" if LANG == "ja" else "Composite score").add_to(m5)
for i, (_, r) in enumerate(g4.nlargest(3, "score_基本").iterrows(), 1):
    c = r.geometry.centroid
    folium.Marker([c.y, c.x], tooltip=f"{i}: {r.nearest_station} {r.dist_station_m}m / score {r['score_基本']:.3f}").add_to(m5)
m5.save(f"cafe_candidates_{LANG}.html"); m5